# 📈 Semana 3: Correlación, Supuestos Estadísticos e Hipótesis

## Objetivos
- Analizar correlaciones entre variables (Pearson, Spearman, Kendall)
- Comprender y evaluar homoscedasticidad y heteroscedasticidad
- Detectar multicolinealidad con VIF
- Formular preguntas de investigación e hipótesis
- Realizar pruebas de hipótesis estadísticas

## Dataset: Water Quality (Calidad del Agua)
Continuamos con el dataset limpio y preparado de la Semana 2.

In [ ]:
# Importación de librerías
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
from scipy import stats
from scipy.stats import pearsonr, spearmanr, kendalltau
import warnings
warnings.filterwarnings('ignore')

# Configuración
plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.max_columns', None)

print("✅ Librerías importadas")

In [ ]:
# Carga y limpieza del dataset (replicando pasos de la Semana 2)
url = "https://raw.githubusercontent.com/jaquimbayoc7/material-fundamentos_datos/main/data/water_potability.csv"
df = pd.read_csv(url)

# Imputar valores faltantes con la mediana
for col in df.columns:
    if df[col].isnull().sum() > 0:
        df[col].fillna(df[col].median(), inplace=True)

# Eliminar duplicados
df = df.drop_duplicates().reset_index(drop=True)

# Variables predictoras
cols_numericas = [c for c in df.select_dtypes(include=[np.number]).columns if c != 'Potability']

print(f"✅ Dataset limpio: {df.shape[0]} filas x {df.shape[1]} columnas")
print(f"Variables predictoras: {cols_numericas}")
df.head()

---
# 1. Análisis de Correlación

La **correlación** mide la fuerza y dirección de la relación **lineal** entre dos variables. Los valores van de -1 a +1:

| Rango | Interpretación |
|-------|----------------|
| 0.00 - 0.19 | Muy débil |
| 0.20 - 0.39 | Débil |
| 0.40 - 0.59 | Moderada |
| 0.60 - 0.79 | Fuerte |
| 0.80 - 1.00 | Muy fuerte |

### Tipos de coeficientes:
- **Pearson (r)**: Mide relación lineal. Asume distribución normal.
- **Spearman (ρ)**: Mide relación monótona. No asume normalidad. Basado en rangos.
- **Kendall (τ)**: Similar a Spearman pero más robusto con muestras pequeñas.

In [ ]:
# ═══════════════════════════════════════════════════════════
# COEFICIENTE DE CORRELACIÓN DE PEARSON
# ═══════════════════════════════════════════════════════════
print("📊 CORRELACIÓN DE PEARSON")
print("=" * 50)

corr_pearson = df[cols_numericas].corr(method='pearson')

plt.figure(figsize=(12, 8))
mask = np.triu(np.ones_like(corr_pearson, dtype=bool))
sns.heatmap(corr_pearson, annot=True, cmap='RdBu_r', fmt='.3f',
            mask=mask, center=0, square=True, linewidths=0.5,
            cbar_kws={'label': 'Coeficiente de Pearson'})
plt.title('Matriz de Correlación de Pearson — Calidad del Agua',
          fontsize=14, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()

In [ ]:
# ═══════════════════════════════════════════════════════════
# COEFICIENTE DE CORRELACIÓN DE SPEARMAN
# ═══════════════════════════════════════════════════════════
print("📊 CORRELACIÓN DE SPEARMAN")
print("=" * 50)

corr_spearman = df[cols_numericas].corr(method='spearman')

plt.figure(figsize=(12, 8))
mask = np.triu(np.ones_like(corr_spearman, dtype=bool))
sns.heatmap(corr_spearman, annot=True, cmap='RdBu_r', fmt='.3f',
            mask=mask, center=0, square=True, linewidths=0.5,
            cbar_kws={'label': 'Coeficiente de Spearman'})
plt.title('Matriz de Correlación de Spearman — Calidad del Agua',
          fontsize=14, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()

In [ ]:
# ═══════════════════════════════════════════════════════════
# COEFICIENTE DE CORRELACIÓN DE KENDALL
# ═══════════════════════════════════════════════════════════
print("📊 CORRELACIÓN DE KENDALL")
print("=" * 50)

corr_kendall = df[cols_numericas].corr(method='kendall')

plt.figure(figsize=(12, 8))
mask = np.triu(np.ones_like(corr_kendall, dtype=bool))
sns.heatmap(corr_kendall, annot=True, cmap='RdBu_r', fmt='.3f',
            mask=mask, center=0, square=True, linewidths=0.5,
            cbar_kws={'label': 'Coeficiente de Kendall'})
plt.title('Matriz de Correlación de Kendall — Calidad del Agua',
          fontsize=14, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()

In [ ]:
# ═══════════════════════════════════════════════════════════
# COMPARACIÓN DE LOS 3 MÉTODOS
# ═══════════════════════════════════════════════════════════
print("📊 COMPARACIÓN DE MÉTODOS DE CORRELACIÓN")
print("=" * 60)

# Seleccionar pares de variables con correlación interesante
pares = [
    ('ph', 'Hardness'),
    ('Solids', 'Conductivity'),
    ('Organic_carbon', 'Trihalomethanes'),
    ('Hardness', 'Sulfate')
]

print(f"{'Par de Variables':35s} {'Pearson':>10s} {'Spearman':>10s} {'Kendall':>10s}")
print("-" * 70)

for var1, var2 in pares:
    r_p, _ = pearsonr(df[var1], df[var2])
    r_s, _ = spearmanr(df[var1], df[var2])
    r_k, _ = kendalltau(df[var1], df[var2])
    print(f"{var1} vs {var2:20s} {r_p:+10.4f} {r_s:+10.4f} {r_k:+10.4f}")

In [ ]:
# Correlación con la variable objetivo (Potability)
print("🎯 CORRELACIÓN CON LA VARIABLE OBJETIVO (Potability)")
print("=" * 55)

corr_target = df[cols_numericas + ['Potability']].corr()['Potability'].drop('Potability').sort_values(ascending=False)

plt.figure(figsize=(10, 5))
colores = ['#2ecc71' if v > 0 else '#e74c3c' for v in corr_target.values]
corr_target.plot(kind='barh', color=colores, edgecolor='white')
plt.xlabel('Coeficiente de Correlación (Pearson)')
plt.title('Correlación de cada Variable con Potabilidad', fontsize=14, fontweight='bold')
plt.axvline(x=0, color='black', linewidth=0.5)
plt.tight_layout()
plt.show()

---
# 2. Homoscedasticidad y Heteroscedasticidad

## Definiciones

- **Homoscedasticidad**: La varianza de los errores (residuos) es **constante** a lo largo de todos los valores de la variable predictora. Es un **supuesto clave** de la regresión lineal.

- **Heteroscedasticidad**: La varianza de los errores **NO es constante** — varía con los valores de la variable predictora. Esto viola el supuesto de regresión y puede producir:
  - Estimadores sesgados de los errores estándar
  - Intervalos de confianza incorrectos
  - Pruebas de hipótesis poco fiables

In [ ]:
from sklearn.linear_model import LinearRegression
import statsmodels.api as sm
from statsmodels.stats.diagnostic import het_breuschpagan, het_goldfeldquandt

# ═══════════════════════════════════════════════════════════
# ANÁLISIS VISUAL DE HOMOSCEDASTICIDAD
# ═══════════════════════════════════════════════════════════
print("📊 ANÁLISIS VISUAL DE RESIDUOS")
print("=" * 50)

# Ajustar un modelo de regresión simple: Hardness → Solids
X = df[['Hardness']].values
y = df['Solids'].values

modelo = LinearRegression()
modelo.fit(X, y)
y_pred = modelo.predict(X)
residuos = y - y_pred

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Scatter con línea de regresión
axes[0].scatter(X, y, alpha=0.3, s=10, color='steelblue')
axes[0].plot(np.sort(X, axis=0), modelo.predict(np.sort(X, axis=0)), color='red', linewidth=2)
axes[0].set_xlabel('Dureza (mg/L)')
axes[0].set_ylabel('Sólidos (ppm)')
axes[0].set_title('Regresión: Dureza → Sólidos', fontweight='bold')

# 2. Residuos vs Valores predichos
axes[1].scatter(y_pred, residuos, alpha=0.3, s=10, color='coral')
axes[1].axhline(y=0, color='black', linewidth=1)
axes[1].set_xlabel('Valores Predichos')
axes[1].set_ylabel('Residuos')
axes[1].set_title('Residuos vs Predichos\n(Verificar homoscedasticidad)', fontweight='bold')

# 3. Distribución de residuos
axes[2].hist(residuos, bins=40, color='mediumseagreen', edgecolor='white', density=True)
axes[2].set_xlabel('Residuos')
axes[2].set_ylabel('Densidad')
axes[2].set_title('Distribución de Residuos', fontweight='bold')

plt.suptitle('Análisis de Residuos — Regresión Lineal', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("\n💡 Si los residuos muestran un patrón de 'embudo' (se amplían o estrechan),")
print("   hay HETEROSCEDASTICIDAD. Si son uniformes, hay HOMOSCEDASTICIDAD.")

In [ ]:
# ═══════════════════════════════════════════════════════════
# PRUEBAS FORMALES DE HETEROSCEDASTICIDAD
# ═══════════════════════════════════════════════════════════
print("🧪 PRUEBAS DE HETEROSCEDASTICIDAD")
print("=" * 55)

# Ajuste con statsmodels para obtener residuos formales
X_sm = sm.add_constant(df['Hardness'])  # Agregar constante (intercepto)
modelo_sm = sm.OLS(df['Solids'], X_sm).fit()

# ── Prueba de Breusch-Pagan ──
print("\n📋 Prueba de Breusch-Pagan")
print("-" * 40)
bp_stat, bp_pvalue, bp_fstat, bp_fpvalue = het_breuschpagan(modelo_sm.resid, X_sm)
print(f"  Estadístico LM: {bp_stat:.4f}")
print(f"  p-valor:        {bp_pvalue:.6f}")
print(f"  Conclusión:     {'❌ HAY heteroscedasticidad' if bp_pvalue < 0.05 else '✅ Homoscedasticidad (no se rechaza H₀)'}")

# ── Prueba de Goldfeld-Quandt ──
print("\n📋 Prueba de Goldfeld-Quandt")
print("-" * 40)
gq_stat, gq_pvalue, gq_order = het_goldfeldquandt(modelo_sm.resid, X_sm)
print(f"  Estadístico F:  {gq_stat:.4f}")
print(f"  p-valor:        {gq_pvalue:.6f}")
print(f"  Conclusión:     {'❌ HAY heteroscedasticidad' if gq_pvalue < 0.05 else '✅ Homoscedasticidad (no se rechaza H₀)'}")

print("\n💡 Regla: Si p-valor < 0.05 → Se rechaza H₀ (homoscedasticidad) → Hay heteroscedasticidad")

In [ ]:
# Análisis de heteroscedasticidad para múltiples relaciones
print("🧪 PRUEBA DE BREUSCH-PAGAN PARA MÚLTIPLES RELACIONES")
print("=" * 65)

resultados_bp = []

for pred in cols_numericas:
    for resp in cols_numericas:
        if pred != resp:
            X_temp = sm.add_constant(df[pred])
            modelo_temp = sm.OLS(df[resp], X_temp).fit()
            _, pval, _, _ = het_breuschpagan(modelo_temp.resid, X_temp)
            resultados_bp.append({
                'Predictora': pred,
                'Respuesta': resp,
                'p-valor': pval,
                'Resultado': 'Heteroscedástico' if pval < 0.05 else 'Homoscedástico'
            })

df_bp = pd.DataFrame(resultados_bp)
hetero_count = (df_bp['Resultado'] == 'Heteroscedástico').sum()
total = len(df_bp)
print(f"\nRelaciones heteroscedásticas: {hetero_count}/{total} ({hetero_count/total*100:.1f}%)")
print("\nEjemplos de relaciones heteroscedásticas:")
display(df_bp[df_bp['Resultado'] == 'Heteroscedástico'].head(10))

### Estrategias para Corregir la Heteroscedasticidad

1. **Transformaciones logarítmicas**: $Y' = \ln(Y)$
2. **Transformación raíz cuadrada**: $Y' = \sqrt{Y}$
3. **Mínimos Cuadrados Ponderados (WLS)**: Dar menos peso a observaciones con mayor varianza
4. **Errores estándar robustos (HAC)**: No corrige la heteroscedasticidad pero ajusta los errores estándar

In [ ]:
# Ejemplo de corrección con transformación logarítmica
print("🔧 CORRECCIÓN CON TRANSFORMACIÓN LOGARÍTMICA")
print("=" * 50)

# Antes: Hardness → Solids (posible heteroscedasticidad)
X_sm = sm.add_constant(df['Hardness'])
modelo_original = sm.OLS(df['Solids'], X_sm).fit()
_, pval_original, _, _ = het_breuschpagan(modelo_original.resid, X_sm)

# Después: Hardness → log(Solids)
df['log_Solids'] = np.log1p(df['Solids'])
modelo_log = sm.OLS(df['log_Solids'], X_sm).fit()
_, pval_log, _, _ = het_breuschpagan(modelo_log.resid, X_sm)

print(f"p-valor Breusch-Pagan (Original):   {pval_original:.6f}")
print(f"p-valor Breusch-Pagan (Log-trans):   {pval_log:.6f}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(modelo_original.fittedvalues, modelo_original.resid, alpha=0.3, s=10, color='coral')
axes[0].axhline(0, color='black')
axes[0].set_title(f'Residuos Original\n(BP p={pval_original:.4f})', fontweight='bold')
axes[0].set_xlabel('Valores Predichos'); axes[0].set_ylabel('Residuos')

axes[1].scatter(modelo_log.fittedvalues, modelo_log.resid, alpha=0.3, s=10, color='mediumseagreen')
axes[1].axhline(0, color='black')
axes[1].set_title(f'Residuos Log-Transformado\n(BP p={pval_log:.4f})', fontweight='bold')
axes[1].set_xlabel('Valores Predichos'); axes[1].set_ylabel('Residuos')

plt.suptitle('Corrección de Heteroscedasticidad con Transformación Log', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
# 3. Multicolinealidad

## Definición

La **multicolinealidad** ocurre cuando dos o más variables predictoras están **altamente correlacionadas** entre sí. Esto causa:
- Inestabilidad en los coeficientes de regresión
- Dificultad para interpretar el efecto individual de cada variable
- Errores estándar inflados

## Detección: Factor de Inflación de Varianza (VIF)

$$VIF_i = \frac{1}{1 - R_i^2}$$

Donde $R_i^2$ es el R² de la regresión de la variable $X_i$ contra todas las demás variables.

| VIF | Interpretación |
|-----|----------------|
| 1 | Sin multicolinealidad |
| 1 - 5 | Moderada |
| 5 - 10 | Alta |
| > 10 | Muy alta (problemática) |

In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

# ═══════════════════════════════════════════════════════════
# CÁLCULO DEL VIF
# ═══════════════════════════════════════════════════════════
print("📐 FACTOR DE INFLACIÓN DE VARIANZA (VIF)")
print("=" * 55)

# Preparar datos (sin la variable objetivo)
X_vif = df[cols_numericas].copy()
X_vif = sm.add_constant(X_vif)

# Calcular VIF para cada variable
vif_data = pd.DataFrame()
vif_data['Variable'] = X_vif.columns[1:]  # Excluir la constante
vif_data['VIF'] = [variance_inflation_factor(X_vif.values, i+1) for i in range(len(cols_numericas))]
vif_data['Interpretación'] = vif_data['VIF'].apply(
    lambda v: '✅ Sin problema' if v < 5 else ('⚠️ Alta' if v < 10 else '❌ Muy alta')
)
vif_data = vif_data.sort_values('VIF', ascending=False)

display(vif_data)

print(f"\n💡 Variables con VIF > 5: {vif_data[vif_data['VIF'] > 5]['Variable'].tolist()}")
print(f"   Variables con VIF > 10: {vif_data[vif_data['VIF'] > 10]['Variable'].tolist()}")

In [ ]:
# Visualización del VIF
plt.figure(figsize=(10, 5))
colores_vif = ['#2ecc71' if v < 5 else '#f39c12' if v < 10 else '#e74c3c' for v in vif_data['VIF']]
plt.barh(vif_data['Variable'], vif_data['VIF'], color=colores_vif, edgecolor='white')
plt.axvline(x=5, color='orange', linestyle='--', label='Umbral moderado (5)')
plt.axvline(x=10, color='red', linestyle='--', label='Umbral alto (10)')
plt.xlabel('VIF', fontsize=12)
plt.title('Factor de Inflación de Varianza (VIF) por Variable', fontsize=14, fontweight='bold')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Detección visual de multicolinealidad con scatter matrix
# Seleccionar las variables con mayor correlación entre sí
corr_abs = corr_pearson.abs()
np.fill_diagonal(corr_abs.values, 0)  # Ignorar diagonal

# Top 5 pares más correlacionados
print("🔗 TOP 5 PARES DE VARIABLES MÁS CORRELACIONADOS")
print("=" * 55)

pares_corr = []
for i in range(len(corr_abs.columns)):
    for j in range(i+1, len(corr_abs.columns)):
        pares_corr.append({
            'Variable 1': corr_abs.columns[i],
            'Variable 2': corr_abs.columns[j],
            '|Correlación|': corr_abs.iloc[i, j]
        })

df_pares = pd.DataFrame(pares_corr).sort_values('|Correlación|', ascending=False)
display(df_pares.head(10))

---
# 4. Definición de la Pregunta de Investigación

## ¿Qué es una Pregunta de Investigación?

Es una pregunta **clara, específica y medible** que guía el análisis de datos. Debe ser:
- **Específica**: Enfocada en un aspecto concreto
- **Medible**: Puede responderse con datos cuantitativos
- **Relevante**: Tiene importancia práctica o científica
- **Factible**: Puede responderse con los datos disponibles

## Preguntas de Investigación para nuestro Dataset

Basándonos en nuestro análisis exploratorio y de correlación, formulamos las siguientes preguntas:

### Pregunta Principal:
> **¿Qué variables fisicoquímicas influyen significativamente en la potabilidad del agua?**

### Preguntas Secundarias:
1. ¿Existe una diferencia significativa en el nivel de pH entre muestras de agua potable y no potable?
2. ¿La dureza del agua está relacionada linealmente con la concentración de sulfatos?
3. ¿La turbidez es un predictor significativo de la potabilidad del agua?

---
# 5. Formulación y Pruebas de Hipótesis

## ¿Qué es una Hipótesis?

- **Hipótesis Nula (H₀)**: Afirmación de que **no hay efecto** o diferencia significativa (lo que queremos refutar)
- **Hipótesis Alternativa (H₁)**: Afirmación de que **sí hay un efecto** o diferencia significativa (lo que queremos demostrar)

### Criterio de decisión:
- Si **p-valor < α** (generalmente 0.05): Rechazamos H₀ → Hay evidencia estadística
- Si **p-valor ≥ α**: No rechazamos H₀ → No hay suficiente evidencia

### Nivel de significancia (α):
- α = 0.05 (5%) es el más común en ciencias
- α = 0.01 (1%) para pruebas más rigurosas

In [ ]:
alpha = 0.05  # Nivel de significancia

# ═══════════════════════════════════════════════════════════
# HIPÓTESIS 1: ¿Hay diferencia en pH entre agua potable y no potable?
# ═══════════════════════════════════════════════════════════
print("🧪 HIPÓTESIS 1: Diferencia de pH por Potabilidad")
print("=" * 55)
print("H₀: No hay diferencia significativa en el pH entre agua potable y no potable")
print("H₁: Hay una diferencia significativa en el pH entre ambos grupos")
print()

# Separar grupos
ph_potable = df[df['Potability'] == 1]['ph']
ph_no_potable = df[df['Potability'] == 0]['ph']

# Prueba t de Student para muestras independientes
t_stat, p_valor = stats.ttest_ind(ph_potable, ph_no_potable)

print(f"  Media pH Potable:    {ph_potable.mean():.4f}")
print(f"  Media pH No Potable: {ph_no_potable.mean():.4f}")
print(f"  Estadístico t:       {t_stat:.4f}")
print(f"  p-valor:             {p_valor:.6f}")
print(f"\n  Decisión: {'❌ RECHAZAR H₀ → Hay diferencia significativa' if p_valor < alpha else '✅ NO RECHAZAR H₀ → No hay evidencia de diferencia'}")

In [ ]:
# Visualización de la Hipótesis 1
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogramas superpuestos
axes[0].hist(ph_no_potable, bins=30, alpha=0.6, color='#e74c3c', label='No Potable', density=True)
axes[0].hist(ph_potable, bins=30, alpha=0.6, color='#2ecc71', label='Potable', density=True)
axes[0].axvline(ph_no_potable.mean(), color='#e74c3c', linestyle='--', linewidth=2)
axes[0].axvline(ph_potable.mean(), color='#2ecc71', linestyle='--', linewidth=2)
axes[0].set_xlabel('pH')
axes[0].set_ylabel('Densidad')
axes[0].set_title('Distribución de pH por Potabilidad', fontweight='bold')
axes[0].legend()

# Boxplot
sns.boxplot(data=df, x='Potability', y='ph', palette=['#e74c3c', '#2ecc71'], ax=axes[1])
axes[1].set_xticklabels(['No Potable', 'Potable'])
axes[1].set_title(f'pH por Potabilidad\n(t={t_stat:.3f}, p={p_valor:.4f})', fontweight='bold')

plt.suptitle('Prueba de Hipótesis: pH vs Potabilidad', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ═══════════════════════════════════════════════════════════
# HIPÓTESIS 2: ¿La dureza está correlacionada con los sulfatos?
# ═══════════════════════════════════════════════════════════
print("🧪 HIPÓTESIS 2: Correlación entre Dureza y Sulfatos")
print("=" * 55)
print("H₀: No existe correlación significativa entre Dureza y Sulfatos (ρ = 0)")
print("H₁: Existe una correlación significativa entre Dureza y Sulfatos (ρ ≠ 0)")
print()

r, p_valor_corr = pearsonr(df['Hardness'], df['Sulfate'])

print(f"  Coeficiente de Pearson (r): {r:.4f}")
print(f"  p-valor:                    {p_valor_corr:.6f}")
print(f"\n  Decisión: {'❌ RECHAZAR H₀ → Correlación significativa' if p_valor_corr < alpha else '✅ NO RECHAZAR H₀ → No hay correlación significativa'}")

In [ ]:
# ═══════════════════════════════════════════════════════════
# HIPÓTESIS 3: ¿Hay diferencia en TODAS las variables entre grupos?
# (ANOVA de un factor para cada variable)
# ═══════════════════════════════════════════════════════════
print("🧪 HIPÓTESIS 3: ANOVA — Diferencias por Potabilidad en todas las variables")
print("=" * 65)
print("H₀: Las medias de ambos grupos (potable vs no potable) son iguales")
print("H₁: Al menos una media es diferente")
print()

resultados_anova = []

for col in cols_numericas:
    grupo_0 = df[df['Potability'] == 0][col]
    grupo_1 = df[df['Potability'] == 1][col]
    f_stat, p_val = stats.f_oneway(grupo_0, grupo_1)
    resultados_anova.append({
        'Variable': col,
        'F-statistic': f_stat,
        'p-valor': p_val,
        'Significativo': '✅ Sí' if p_val < alpha else '❌ No'
    })

df_anova = pd.DataFrame(resultados_anova).sort_values('p-valor')
display(df_anova)

variables_sig = df_anova[df_anova['p-valor'] < alpha]['Variable'].tolist()
print(f"\n📋 Variables con diferencia significativa (p < {alpha}): {variables_sig if variables_sig else 'Ninguna'}")

In [ ]:
# Resumen visual de todas las pruebas de hipótesis
plt.figure(figsize=(12, 5))

colores_pval = ['#2ecc71' if p < alpha else '#e74c3c' for p in df_anova['p-valor']]
plt.barh(df_anova['Variable'], -np.log10(df_anova['p-valor']), color=colores_pval, edgecolor='white')
plt.axvline(x=-np.log10(alpha), color='red', linestyle='--', label=f'Umbral α={alpha} (-log₁₀={-np.log10(alpha):.2f})')
plt.xlabel('-log₁₀(p-valor)', fontsize=12)
plt.title('Significancia Estadística por Variable (ANOVA)',
          fontsize=14, fontweight='bold')
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()

print("💡 Barras VERDES (a la derecha de la línea roja) = Variables con diferencia significativa")

---
## Conclusiones

En esta semana aprendimos:

1. **Correlación**: Analizamos relaciones lineales (Pearson) y monótonas (Spearman, Kendall) entre las variables de calidad del agua. Encontramos que las correlaciones son generalmente débiles.

2. **Homoscedasticidad/Heteroscedasticidad**: Evaluamos la constancia de la varianza de los residuos usando pruebas formales (Breusch-Pagan, Goldfeld-Quandt) y gráficos de residuos. Aplicamos transformaciones logarítmicas como estrategia de corrección.

3. **Multicolinealidad**: Calculamos el VIF para detectar variables predictoras altamente correlacionadas entre sí. Identificamos las variables que podrían causar problemas en modelos de regresión.

4. **Preguntas de investigación**: Formulamos preguntas claras y medibles sobre las relaciones entre variables fisicoquímicas y potabilidad.

5. **Hipótesis**: Construimos y probamos hipótesis estadísticas usando t-test, correlación de Pearson y ANOVA. Determinamos qué variables presentan diferencias significativas entre agua potable y no potable.

**Próxima semana:** Construiremos modelos de Regresión Lineal Simple y Múltiple para predecir variables de calidad del agua, aplicando todos los conocimientos adquiridos.